In [ ]:
import folium
from datetime import datetime
import pandas as pd
import numpy as np
import tqdm 
from matplotlib import pyplot as plt
import config
import pickle
import networkx as nx
import requests
import time 
from ast import literal_eval
from joblib import Parallel, delayed
from routingpy.routers import get_router_by_name


In [7]:
router = get_router_by_name("osrm")(base_url="http://router.project-osrm.org")

In [8]:
GENERATE_HEATMAP_EDGES = False
GENERATE_HEATMAP_PATHS = True
COMPUTE_DISTANCE_MATRIX = False 
COMPUTE_STATION_DF = False 

# Query information from Open Street Maps API

In [4]:
def query_wrapper(orig, dest, invert=True):
    if invert:
        return router.directions([(orig[1], orig[0]), (dest[1], dest[0])], 'driving')
    else:
        return router.directions(locations=[orig, dest], profile='driving')
    

In [5]:
query_wrapper((42.393167, -71.064352),(40.718037, -73.932309))

Direction([(-71.06449, 42.39304), (-71.12776, 42.3568), (-71.31359, 42.33493), (-71.36878, 42.31132), (-71.4126, 42.3154), (-71.55771, 42.26972), (-71.61863, 42.2294), (-71.75598, 42.2211), (-72.06132, 42.13087), (-72.20559, 41.94815), (-72.37392, 41.85737), (-72.4308, 41.85429), (-72.55391, 41.80158), (-72.57655, 41.77512), (-72.65793, 41.75046), (-72.64409, 41.69919), (-72.67389, 41.64735), (-72.72304, 41.57225), (-72.82142, 41.48151), (-72.87593, 41.39228), (-73.03055, 41.31074), (-73.07877, 41.24942), (-73.17964, 41.22199), (-73.19417, 41.17072), (-73.32095, 41.12104), (-73.42849, 41.10481), (-73.46794, 41.0746), (-73.72306, 40.97402), (-73.83566, 40.85296), (-73.83562, 40.74661), (-73.93223, 40.71785)], 16528, 340162)

In [6]:
def bat_consumption(dist, time, start_elev_m, end_elev_m, start_speed, end_speed):
    # To compute the reduction in SOC from this output: if X returned,
    # X * (2.7778 x 10^-7 J/kwh)/(battery capacity in kwh)
    
    m = 24493.988 #kg
    Af = 3.825 #m^2
    Cpi = 0.011
    Cd = 0.9
    nt = .9 # transmission efficiency of battery
    nmd = .85 # electrical machine efficiency, also called motor efficiency
    nr = .05 # regenerative braking efficiency
    rho = 1.225 #kg/m
    g = 9.8 #m/s^2
    V = dist/time
    dVdt = (end_speed - start_speed)/time
    alpha = (end_elev_m - start_elev_m)/dist # all units in meters
    nwh = 1
    
    W_acc = 0 # accessory load
    
    W_tract = m * V * dVdt + .5 * Cd * Af * rho * V**3 + m * g * V * Cpi +  m * g * V * np.sin(alpha)
    
    if W_tract > 0:
        W_dis = (W_tract + W_acc)/(nmd * nt)
    else:
        W_dis = W_acc

    if W_tract < 0:
        W_chg = - 1* W_acc + abs(W_tract)/nwh * nmd * nt
    else:
        W_chg = 0
    
    return (W_dis - W_chg) * time
    

def get_elevation(locations, starting_j=0):
    """
    Returns elevation of list of locations (each location is a tuple (lat, long))
    
    starting_j allows for a subset of this list to be queried (primarily used for recovery if code crashes in-between)
    """
    elevation_dict = dict()
    N = len(locations)
    for j in tqdm.tqdm(range(starting_j, N)): 
        x = locations[j]
        req_str = 'https://api.open-elevation.com/api/v1/lookup?locations='
        req_str += str(x[0]) +"," +  str(x[1])
        
        r = requests.get(req_str, timeout=3600)

        if r.status_code == 200: # successful query
            for x in r.json()['results']:
                elevation_dict[(np.round(x['latitude'], 6), np.round(x['longitude'], 6))] = x['elevation']
        
        elif r.status_code == 504: # likely too many queries were made in a short period of time
            print("sleeping")
            time.sleep(30)
            r = requests.get(req_str, timeout=3600)
                             
        else: # some other error has occured
            print(r.content)
            return elevation_dict, j

    return elevation_dict

def get_data(orig, dest, elev):
    try:
        P = query_wrapper(orig, dest)
        time.sleep(1)
    except:
        return np.inf, np.inf, np.inf
    
    # Code to approximate starting and ending speed (used for battery consumption computation)
    j = 0
    dur = 0
    while dur == 0:
        j += 1
        first_step = query_wrapper(P.geometry[0], P.geometry[j], invert=False)
        time.sleep(1)
        dur = first_step.duration
    j = 0
    dur = 0
    while dur == 0:
        j += 1
        last_step = query_wrapper(P.geometry[0 - j - 1], P.geometry[-1], invert=False)
        dur = last_step.duration
    v0 = first_step.distance/first_step.duration
    v1 = last_step.distance/last_step.duration
    
    
    return P.distance, P.duration, bat_consumption(P.distance, P.duration, elev[orig], elev[dest], v0, v1)
    



hello
hello


# Filtering charging station dataset

In [9]:
stations = pd.read_csv("data/all_fuel_stations.csv")[['Latitude', 'Longitude']].values.tolist()

# Computing Distance Matrix 

In [10]:
with open("data/pickled/v2_elev.pickle", 'rb') as file:
    elev_dict = pickle.load(file)

#elev_dict = get_elevation(stations, 0)


 11%|█████████████████▊                                                                                                                                                    | 77/718 [00:54<07:37,  1.40it/s]


 22%|███████████████████████████████████▌                                                                                                                                 | 155/718 [01:50<06:10,  1.52it/s]


 32%|█████████████████████████████████████████████████████▌                                                                                                               | 233/718 [02:45<06:04,  1.33it/s]


 43%|███████████████████████████████████████████████████████████████████████▍                                                                                             | 311/718 [03:41<04:14,  1.60it/s]


 54%|█████████████████████████████████████████████████████████████████████████████████████████▍                                                                           | 389/718 [04:36<03:55,  1.40it/s]


 65%|███████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                         | 467/718 [05:31<02:52,  1.45it/s]


 76%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                       | 545/718 [06:27<02:11,  1.31it/s]


 87%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                     | 623/718 [07:24<01:01,  1.54it/s]


 98%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████    | 701/718 [08:21<00:12,  1.38it/s]


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 718/718 [08:35<00:00,  1.39it/s]


In [8]:
if COMPUTE_DISTANCE_MATRIX:
    n = len(stations)
    def process(i): 
            try:
                x = (np.round(stations[i][0], 6), np.round(stations[i][1], 6))
                nbrs = [j for j in range(n) if abs(x[0] - stations[j][0]) <= 4 and abs(x[1] - stations[j][1]) <= 5]
                res = []
                for j in nbrs:
                    if i == j:
                        continue
                    y = (np.round(stations[j][0], 6), np.round(stations[j][1], 6))
                    val = get_data(x, y, elev_dict)
                    res.append({'pt1': x, 'pt2': y, 'dist': val[0], 'time': val[1], 'bat': val[2]})
                return res 
            except:
                print("ERROR: ", i)
                return None

    records = [process(i) for i in tqdm.tqdm(range(n))]
    records = [x for x in records if x is not None]
    pd.DataFrame(records).to_csv("D_parallel_out.csv")


# Plotting Heatmap of Power Demand

In [ ]:
MAX_LAT = max([x[0] for x in stations_to_keep])
MIN_LAT = min([x[0] for x in stations_to_keep])
MAX_LNG = max([x[1] for x in stations_to_keep])
MIN_LNG = min([x[1] for x in stations_to_keep])

In [ ]:
def c(x):
    """
    Explanation: To convert to kw, we divide bat_consumption (in J/s) by 100. 
    
    Ideally, (bat consumption) = [a fixed constant (2880)] * (time); while this doesn't hold exactly,
    the following heatmap shows that it holds within a multiplicative factor
  
    
    """ 
    kw = x/(1000)
    y = kw/2880
    if 0 <= y < 1:
        return "#33FFFF"
    elif 1 <= y < 10:
        return "#33C4FF"
    elif 10 <= y < 20:
        return "#3399FF"
    elif 20 <= y <= 30:
        return "#7300e6"
    elif y > 30:
        return "#1ced38"
    else:
        return "#FF5B33"
    
fmap = folium.Map(location = ((config.MAX_LAT + config.MIN_LAT)/2, (config.MAX_LNG + config.MIN_LNG)/2), zoom_start=5)

if GENERATE_HEATMAP_EDGES:    
    bat_consumptions = []

    n = len(stations_to_keep)


    stations_to_keep = [tuple(x) for x in stations_to_keep]
    for i in range(n):
        for j in tqdm.tqdm(range(i+1, n)):
            x = stations_to_keep[i]
            y = stations_to_keep[j]

            P = D[x][y]
            if P == np.inf:
                continue
            lenP = len(P['edges'])
            for r in range(lenP):
                e = P['edges'][r]
                start = P['nodes'][r]
                end = P['nodes'][r + 1]

                snapped_path=gmaps.snap_to_roads([(start['lat'], start['lng']), (end['lat'], end['lng'])], interpolate=True)
                to_plot = [(start['lat'], start['lng'])] + [(x['location']['latitude'], x['location']['longitude']) for x in snapped_path] + [ (end['lat'], end['lng'])]

                bat = bat_consumption(e[2]['speed'], start['elev'], end['elev'], e[2]['dist'] * 1000)
                bat_consumptions.append(bat/e[2]['time'])
                folium.PolyLine(locations=to_plot, color=c(bat)).add_to(fmap)
        fmap.save("heatmap.html")
            
fmap

In [ ]:
fmap

In [9]:
csv = pd.read_csv("D_parallel_out.csv")
csv = csv[[str(i) for i in range(147)]]
csv = csv.replace(np.NaN, None)

In [10]:
records = []
inf = np.inf 
for __, row in csv.iterrows():
    for x in row:
        if x is not None:
            x = x.replace('inf', 'None')
            try:
                records.append(literal_eval(x))
            except:
                print(x)
                break
df = pd.DataFrame(records)

In [11]:
print(df)
df.to_csv("data/D_clean_Jun12.csv")

                           pt1                      pt2      dist     time  \
0      (42.393167, -71.064352)  (40.718037, -73.932309)  340162.0  16527.0   
1      (42.393167, -71.064352)  (41.527246, -72.064419)  159539.0   7528.0   
2      (42.393167, -71.064352)  (42.439859, -71.257005)   20824.0   1395.0   
3      (42.393167, -71.064352)  (44.461715, -73.215217)  345032.0  15362.0   
4      (42.393167, -71.064352)  (39.417088, -74.538731)  535266.0  24756.0   
...                        ...                      ...       ...      ...   
56283   (34.07523, -117.29432)    (36.97874, -121.9536)  634116.0  27097.0   
56284   (34.07523, -117.29432)   (37.86697, -122.26723)  689711.0  28930.0   
56285   (34.07523, -117.29432)   (35.46767, -119.07412)  277390.0  11960.0   
56286   (34.07523, -117.29432)   (33.89859, -118.27977)  108732.0   4904.0   
56287   (34.07523, -117.29432)    (33.7441, -118.19161)  115411.0   5258.0   

                bat  
0      1.577455e+09  
1      7.534472e+08

In [12]:
LA = (34.01665, -118.208679)
NW = (40.67626, -74.24808)

In [35]:
from ast import literal_eval
fmap = folium.Map(location = ((config.MAX_LAT + config.MIN_LAT)/2, (config.MAX_LNG + config.MIN_LNG)/2), zoom_start=5)

if GENERATE_HEATMAP_PATHS:
    G = nx.DiGraph()
    for ind, row in df.iterrows():
        x = row['pt1']
        y = row['pt2'] 
        G.add_edge(x, y, dist=row['dist'], time=row['time'], bat=row['bat'])
             


    for x in nx.shortest_path(G, LA, NW, weight='bat'):
        for y in G.neighbors(x):
            print(y)
            try:
                SP = nx.shortest_path(G, x, y, weight="bat") # shortest charge path
                if sum(G[SP[i]][SP[i+1]]['time'] for i in range(len(SP)-1)) <= 1.10 * nx.shortest_path_length(G, x, y, weight="time"):
                    #folium.PolyLine(locations=nx.shortest_path(G, x, y, weight="time"), color="gray").add_to(fmap)
                    continue
                else:
                    folium.PolyLine(locations=nx.shortest_path(G, x, y, weight="time"), dash_array='10', color="red").add_to(fmap)
                    folium.PolyLine(locations=SP, color="#3399FF", dash_array='17').add_to(fmap)
            except:
                continue
 

        fmap.save("heatmap_path.html")
            
fmap

(35.317617, -119.039048)
(36.72136, -119.761165)
(37.642339, -122.117821)
(37.990624, -121.283787)
(32.631042, -117.057953)
(32.978339, -117.034924)
(33.862772, -117.234706)
(34.025657, -118.031019)
(33.968221, -118.109093)
(34.209219, -118.496403)
(34.091688, -117.87986)
(34.23702, -118.597759)
(33.676026, -117.764558)
(36.220567, -115.125553)
(33.899899, -118.207088)
(34.047068, -117.634077)
(34.042311, -117.584572)
(36.169487, -119.338155)
(34.508713, -117.332343)
(34.421443, -119.684178)
(36.590599, -119.4258)
(35.75191, -119.240762)
(34.539391, -117.2962)
(36.76452, -119.8305)
(37.9816, -122.04593)
(33.887527, -117.571533)
(33.923992, -117.411949)
(37.632815, -122.398352)
(34.138664, -117.923766)
(33.860704, -117.865217)
(34.689589, -118.130371)
(34.118101, -116.45646)
(37.73137, -122.212097)
(37.88175, -122.306259)
(34.886448, -117.079423)
(33.807567, -118.284438)
(33.810331, -118.162929)
(33.703906, -117.863732)
(37.746023, -122.188974)
(37.798995, -122.28338)
(37.299692, -120.4

(38.576871, -121.570377)
(34.137674, -116.303336)
(35.400164, -119.399727)
(33.952886, -118.377315)
(33.839783, -117.254204)
(34.919891, -120.47825)
(34.063139, -117.61148)
(33.933343, -118.370006)
(34.660758, -120.452355)
(34.05305, -118.116522)
(33.545881, -117.191846)
(38.572721, -121.546207)
(34.060637, -117.488248)
(33.919681, -117.940717)
(37.630952, -120.918922)
(34.097359, -117.37883)
(34.069321, -117.45157)
(33.463568, -112.459255)
(33.904195, -116.552357)
(35.210277, -118.832618)
(35.443791, -119.077703)
(34.247415, -118.405835)
(35.61701, -119.658905)
(35.125636, -118.408202)
(36.352663, -119.42401)
(33.928312, -117.558425)
(33.921013, -118.221077)
(33.832757, -118.235208)
(37.275418, -121.866073)
(37.31412, -121.86017)
(33.888576, -117.275543)
(34.065605, -117.554952)
(37.110094, -121.015475)
(34.425432, -117.291142)
(36.34122, -119.38985)
(33.7822, -118.2108)
(35.6157, -119.65721)
(34.00562, -118.12381)
(37.93507, -121.31174)
(34.56089, -120.14103)
(35.97851, -114.83517)
(

(35.490972, -98.967096)
(35.838731, -94.631014)
(35.106478, -92.43572)
(37.737099, -97.32375)
(32.645801, -96.840907)
(32.598649, -97.317542)
(32.4532, -94.706348)
(35.395749, -97.267868)
(32.917907, -96.419128)
(32.448848, -100.439583)
(32.441468, -93.765558)
(38.89357, -94.36045)
(38.971499, -92.294316)
(38.96627, -95.700495)
(34.071417, -99.009382)
(32.824385, -97.336689)
(37.199194, -93.402643)
(35.335799, -94.384145)
(32.627591, -97.314361)
(32.789008, -96.652678)
(32.354553, -95.302495)
(35.309665, -99.631741)
(36.249164, -95.333829)
(32.181782, -94.334174)
(36.127057, -96.352955)
(35.58558, -97.58744)
(32.698465, -97.959695)
(39.088255, -94.732123)
(32.693705, -96.910233)
(32.624236, -96.843053)
(39.149026, -94.617479)
(37.962455, -100.846437)
(36.113892, -95.889587)
(39.13353, -94.523119)
(36.535661, -97.33916)
(34.188212, -97.171022)
(35.486391, -95.148721)
(35.201617, -91.732791)
(35.150538, -97.479381)
(36.167392, -94.119169)
(35.02927, -94.647628)
(35.626543, -97.496058)
(3

(35.201617, -91.732791)
(36.167392, -94.119169)
(35.02927, -94.647628)
(39.091195, -94.682103)
(41.738117, -86.333532)
(38.797138, -90.076424)
(41.805516, -87.818549)
(39.446197, -87.128195)
(35.165628, -86.535068)
(41.571315, -87.653619)
(42.47014, -92.44708)
(42.511462, -90.656075)
(41.768307, -88.191536)
(41.587098, -87.302188)
(40.46255, -86.11641)
(40.31694, -94.87345)
(39.48295, -88.37203)
(40.42523, -86.90953)
(40.421027, -86.917922)
(40.4297, -86.91118)
(40.42231, -86.91705)
(39.93482, -86.02704)
(39.87052, -86.14224)
(42.0898, -88.24391)
(39.834688, -86.25018)
(41.622812, -87.527426)
(39.930318, -82.885496)
(41.402106, -81.828365)
(40.093708, -83.162608)
(41.822729, -87.749999)
(42.724204, -87.952181)
(40.853115, -81.769421)
(39.374016, -84.547382)
(41.010989, -87.276086)
(40.074011, -86.9067)
(38.134634, -85.737722)
(40.70153, -89.642572)
(38.373132, -85.747053)
(43.058845, -88.204801)
(43.043029, -89.352921)
(42.22456, -89.109935)
(40.185035, -86.53319)
(39.699896, -86.31425

(41.966439, -76.534294)
(40.087252, -76.095113)
(42.102397, -75.839707)
(40.860729, -75.27151)
(42.922551, -78.628437)
(40.191991, -74.794534)
(40.387923, -74.085991)
(40.247319, -74.24322)
(39.989815, -74.258336)
(41.554634, -72.107836)
(41.3634, -75.730103)
(41.294864, -72.899122)
(42.983441, -78.91425)
(44.590314, -69.289139)
(40.27102, -78.83327)
(40.450231, -76.415776)
(39.660766, -75.693486)
(39.317115, -76.531988)
(40.055995, -76.327723)
(41.315531, -75.753889)
(40.724957, -76.317177)
(42.938929, -78.823213)
(43.062221, -73.822235)
(40.744271, -75.218923)
(40.838387, -77.787831)
(39.821725, -75.240765)
(41.125042, -78.765941)
(39.974852, -76.723959)
(40.23601, -77.122908)
(42.128286, -79.239695)
(39.497643, -76.169957)
(42.251305, -71.805502)
(43.970783, -70.609685)
(43.139294, -77.666717)
(43.120271, -76.220444)
(40.805695, -73.880803)
(40.334523, -78.900235)
(39.944688, -76.793636)
(38.472406, -78.878672)
(36.755694, -76.224619)
(37.39664, -79.229086)
(43.182787, -71.495001)
(

In [13]:
fmap

NameError: name 'fmap' is not defined

# Computing Station_DF

In [10]:
electric_stations = pd.read_csv("fuel_station_locations.csv")[['Latitude', 'Longitude']].values.tolist()

def snap_to_stations(pt):
    candidate_stat =  [x for x in electric_stations if abs(x[0] - pt[0]) <= 1 and abs(x[1] - pt[1]) <= 1]
    if len(candidate_stat) == 0:
        return None, np.inf
    closest = None
    closest_val = np.inf
    for x in candidate_stat:
        dist = query_wrapper(x, pt).distance
        if dist < closest_val:
            closest = x
            closest_val = dist
    
    return closest, closest_val

In [11]:
from joblib import Parallel, delayed
def process(line):
    res = []
    parts = line.strip().split(',')
    lat = float(parts[0])/(10**7)
    lng = float(parts[1])/(10**7) 
    if not (config.MIN_LAT <= lat <= config.MAX_LAT and config.MIN_LNG <= lng <= config.MAX_LNG):
        return None  

    try:
        closest_stat, dist = snap_to_stations((lat, lng))
        station_info = dict([('lat',lat ), ('lng', lng), ('total_chargers_at_stat', int(parts[2])), ('snap_to', closest_stat), ('dist_snap', dist)])

        charger_types = dict()
        i = 3
        while i < len(parts) - 1:
            charger = int(parts[i])
            if charger in charger_types:
                charger_types[charger]["n_chargers"] += 1
            else:
                charger_types[charger] = {'n_chargers': 1, 'kw': float(parts[i+1])}
            i += 2
        for t in charger_types.keys():
            node_info = dict(list(station_info.items()) + [('type', t)] + list(charger_types[t].items()))
            res.append(node_info)

    except:
        print("ERROR OCCURRED")
        res = None
    
    return res

records =  Parallel(n_jobs=10)(delayed(process)(i) for i in tqdm.tqdm(open('relevant_stat_data.txt')))

11640it [00:00, 29960.12it/s]


ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCURRED
ERROR OCCU

In [12]:
stations_df = pd.DataFrame(sum([x for x in records if x is not None], [])).dropna().reset_index(drop=True)
stations_df['index'] = stations_df.index
print(stations_df)
print(stations_df['snap_to'])
stations_df['snap_to'] = stations_df['snap_to'].apply(lambda x: tuple(x))

Empty DataFrame
Columns: [lat, lng, total_chargers_at_stat, snap_to, dist_snap, type, n_chargers, kw, index]
Index: []
Series([], Name: snap_to, dtype: object)


In [ ]:
# Keeping only the closest points to each station
snap_dist = dict()
for stat, stat_df in stations_df.groupby('snap_to'):
    snap_dist[stat] = stat_df.sort_values(by='dist_snap').iloc[0]
rows = [v['index'] for k,v in snap_dist.items()] # if v['dist_snap'] <= 20]
stations_df = stations_df.loc[rows]
print(stations_df.shape)

In [ ]:
stations_df

In [ ]:
stations_df.to_csv("processed.csv", index=True)

In [ ]:
snap_to_stations((34.017194, -118.263477)) #LA

In [ ]:
snap_to_stations((40.738932, -74.175708)) #Newark